# NYC 311 — Data Cleaning

This notebook cleans and validates the raw NYC 311 service request
snapshot before feature engineering and machine learning.

Input:
- data/raw/nyc311_2025_sample.csv

Output:
- data/processed/nyc311_2025_cleaned.csv

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "nyc311_2025_sample.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "nyc311_2025_cleaned.csv"

In [5]:
print(RAW_PATH)
print(RAW_PATH.exists())

/Users/mahimjohn/Desktop/NYC311-Triage-capstone-project/data/raw/nyc311_2025_sample.csv
True


In [6]:
raw = pd.read_csv(RAW_PATH)

print("Rows:", raw.shape[0])
print("Columns:", raw.shape[1])

Rows: 100000
Columns: 13


In [7]:
required_columns = [
    "unique_key",
    "created_date",
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "borough",
    "city",
    "latitude",
    "longitude",
    "closed_date"
]

missing_columns = [
    column for column in required_columns
    if column not in raw.columns
]

unexpected_columns = [
    column for column in raw.columns
    if column not in required_columns
]

print("Missing required columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing required columns: []
Unexpected columns: []


In [8]:
df = raw.copy()

In [9]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])

In [10]:
print(df[["created_date", "closed_date"]].dtypes)

created_date    datetime64[ns]
closed_date     datetime64[ns]
dtype: object


In [11]:
duplicate_keys = df["unique_key"].duplicated().sum()

print("Duplicate unique keys:", duplicate_keys)

Duplicate unique keys: 0


In [12]:
categorical_columns = [
    "agency",
    "agency_name",
    "complaint_type",
    "descriptor",
    "location_type",
    "borough",
    "city"
]

for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")

In [13]:
df[categorical_columns].isna().sum()

agency            0
agency_name       0
complaint_type    0
descriptor        0
location_type     0
borough           0
city              0
dtype: int64

In [14]:
df["resolution_days"] = (
    df["closed_date"] - df["created_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [15]:
df["resolution_days"].describe()

count    100000.000000
mean          8.386056
std          32.876383
min          -2.243750
25%           0.078238
50%           0.509508
75%           1.990162
max         603.805347
Name: resolution_days, dtype: float64

In [16]:
invalid_duration = df["resolution_days"] < 0

print("Invalid resolution records:", invalid_duration.sum())

Invalid resolution records: 2


In [17]:
df = df[~invalid_duration].copy()

In [18]:
print("Negative resolution times remaining:",
      (df["resolution_days"] < 0).sum())

Negative resolution times remaining: 0


In [19]:
df["delay_flag"] = (df["resolution_days"] > 7).astype(int)

In [20]:
print(df["delay_flag"].value_counts())
print()
print(df["delay_flag"].value_counts(normalize=True) * 100)

delay_flag
0    87947
1    12051
Name: count, dtype: int64

delay_flag
0    87.948759
1    12.051241
Name: proportion, dtype: float64


In [21]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate unique keys:",
      df["unique_key"].duplicated().sum())

print("\nNegative resolution times:",
      (df["resolution_days"] < 0).sum())

Rows: 99998
Columns: 15

Missing values:
unique_key           0
created_date         0
agency               0
agency_name          0
complaint_type       0
descriptor           0
location_type        0
incident_zip       473
borough              0
city                 0
latitude           794
longitude          794
closed_date          0
resolution_days      0
delay_flag           0
dtype: int64

Duplicate unique keys: 0

Negative resolution times: 0


In [23]:
print("Missing incident_zip:", df["incident_zip"].isna().sum())

print("Missing latitude:", df["latitude"].isna().sum())

print("Missing longitude:", df["longitude"].isna().sum())

print(
    "Missing both latitude and longitude:",
    (df["latitude"].isna() & df["longitude"].isna()).sum()
)

print(
    "Missing latitude but longitude present:",
    (df["latitude"].isna() & df["longitude"].notna()).sum()
)

print(
    "Missing longitude but latitude present:",
    (df["longitude"].isna() & df["latitude"].notna()).sum()
)

Missing incident_zip: 473
Missing latitude: 794
Missing longitude: 794
Missing both latitude and longitude: 794
Missing latitude but longitude present: 0
Missing longitude but latitude present: 0


## Cleaning Summary

The raw NYC 311 dataset contained 100,000 service requests.

The following cleaning operations were performed:

- Duplicate request IDs were checked and no duplicates were found.
- Date fields were converted to datetime format.
- Missing categorical values were replaced with "Unknown".
- Two records with negative resolution times were identified as invalid and removed.
- Missing geographic values were retained for handling during model preprocessing.
- Resolution time was calculated from `created_date` and `closed_date`.
- A binary `delay_flag` target was created using a 7-day resolution threshold.

The resulting cleaned dataset contains 99,998 records and 15 columns.

In [24]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_PATH, index=False)

print("Cleaned dataset saved to:")
print(PROCESSED_PATH)

Cleaned dataset saved to:
/Users/mahimjohn/Desktop/NYC311-Triage-capstone-project/data/processed/nyc311_2025_cleaned.csv


## Initial Feature Selection

In [27]:
feature_selection = pd.DataFrame({
    "feature": [
        "agency",
        "complaint_type",
        "borough",
        "descriptor",
        "location_type",
        "city",
        "incident_zip",
        "latitude",
        "longitude",
        "created_date",
        "unique_key",
        "agency_name",
        "closed_date",
        "resolution_days",
        "delay_flag"
    ],
    "decision": [
        "Selected",
        "Selected",
        "Selected",
        "Later",
        "Later",
        "Later",
        "Later",
        "Later",
        "Later",
        "Feature engineering later",
        "Exclude",
        "Exclude",
        "Exclude - target leakage",
        "Exclude - target leakage",
        "Target"
    ],
    "reason": [
        "Low cardinality and meaningful variation in delay rates",
        "Important request category with variation in delay rates",
        "Provides geographic context",
        "High cardinality; evaluate later",
        "Many categories and missing values",
        "Higher cardinality; evaluate later",
        "Geographic variable requiring preprocessing",
        "Geographic variable with missing values",
        "Geographic variable with missing values",
        "Can be transformed into prediction-time features",
        "Identifier only",
        "Duplicates information represented by agency",
        "Known only after resolution",
        "Directly derived from closed and created dates",
        "Prediction target"
    ]
})

feature_selection

,feature,decision,reason
0,agency,Selected,Low cardinality and meaningful variation in de...
1,complaint_type,Selected,Important request category with variation in d...
2,borough,Selected,Provides geographic context
3,descriptor,Later,High cardinality; evaluate later
4,location_type,Later,Many categories and missing values
5,city,Later,Higher cardinality; evaluate later
6,incident_zip,Later,Geographic variable requiring preprocessing
7,latitude,Later,Geographic variable with missing values
8,longitude,Later,Geographic variable with missing values
9,created_date,Feature engineering later,Can be transformed into prediction-time features


In [28]:
experiment_1_features = [
    "agency",
    "complaint_type",
    "borough"
]

experiment_1_target = "delay_flag"

experiment_1_data = df[
    experiment_1_features + [experiment_1_target]
].copy()

experiment_1_data.head()

,agency,complaint_type,borough,delay_flag
0,NYPD,Noise - Residential,BRONX,0
1,NYPD,Noise - Residential,BROOKLYN,0
2,NYPD,Noise - Residential,BRONX,0
3,NYPD,Illegal Fireworks,BROOKLYN,0
4,HPD,HEAT/HOT WATER,BRONX,0


In [29]:
print("Experiment 1 shape:", experiment_1_data.shape)

print("\nFeatures:")
print(experiment_1_features)

print("\nTarget:")
print(experiment_1_target)

Experiment 1 shape: (99998, 4)

Features:
['agency', 'complaint_type', 'borough']

Target:
delay_flag


In [30]:
print(experiment_1_data.isna().sum())

agency            0
complaint_type    0
borough           0
delay_flag        0
dtype: int64
